## 🏠 metodo_workflow

In [1]:
import pandas as pd
import janitor  # pyjanitor se importa así para extender las funcionalidades de pandas
import numpy as np # Necesario para crear valores nulos (NaN)

# --- 1. Construir un DataFrame de Prueba más Complejo ---
# Ahora incluimos valores nulos (NaN) y una fila completamente vacía
# para simular datos aún más "sucios".
data = {
    'ID de Transacción': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111],
    'nombre_cliente': ['Maria', 'Juan', 'Pedro', 'Maria', 'Juan', 'Ana', 'Maria', 'Juan', None, 'Carlos', None],
    'apellido_cliente': ['Gomez', 'Perez', 'Lopez', 'Gomez', 'Perez', 'Silva', 'Gomez', 'Perez', 'Ramirez', None, None],
    'monto': [100, 150, 200, 100, 250, 80, 100, 150, 50, 300, np.nan],
    'fecha': ['2025-09-10', '2025-09-10', '2025-09-11', '2025-09-10', '2025-09-12', '2025-09-12', '2025-09-13', '2025-09-10', '2025-09-14', '2025-09-15', None]
}
df_sucio = pd.DataFrame(data)

print("--- 1. DataFrame Original (Sucio con Nulos y Duplicados) ---")
print("Observa los valores 'NaN' y 'None', además de los duplicados.")
print(df_sucio)
print("\n" + "="*60 + "\n")


# --- 2. Identificando los Problemas ---
# Un buen analista primero diagnostica el alcance de los datos sucios.
print("--- 2a. Identificando Duplicados LÓGICOS (por nombre y apellido) ---")
duplicados_logicos = df_sucio.get_dupes(column_names=['nombre_cliente', 'apellido_cliente'])
print(duplicados_logicos)
print("\n" + "--- 2b. Identificando Filas con Nulos en Columnas Clave ---")
filas_con_nulos = df_sucio[df_sucio['nombre_cliente'].isna() | df_sucio['apellido_cliente'].isna()]
print(filas_con_nulos)
print("\n" + "="*60 + "\n")


# --- 3. Flujo de Trabajo de Limpieza Encadenado con Pyjanitor ---
# Realizamos TODA la limpieza en una sola secuencia de operaciones lógicas.
# El orden de las operaciones es importante.

print("--- 3. Proceso de Limpieza Encadenado (Nulos y Duplicados) ---")
df_limpio = (
    df_sucio
    .clean_names()  # 1. Estandarizar nombres de columnas (ej: 'ID de Transacción' -> 'id_de_transaccion')
    .remove_empty() # 2. Eliminar filas y columnas que estén COMPLETAMENTE vacías.
    .dropna(subset=['nombre_cliente', 'apellido_cliente']) # 3. Eliminar filas donde el nombre O el apellido son nulos.
    .drop_duplicates(subset=['nombre_cliente', 'apellido_cliente'], keep='first') # 4. Eliminar registros duplicados de clientes.
    .reset_index(drop=True) # 5. Reconstruir el índice del DataFrame para que sea secuencial.
)

print("DataFrame Limpio (sin nulos en columnas clave y sin duplicados):")
print(df_limpio)
print("\n" + "="*60 + "\n")


# --- 4. Verificación Final ---
# Comprobamos que ambos problemas (nulos y duplicados) han sido resueltos.
print("--- 4. Verificación Post-Limpieza ---")
duplicados_restantes = df_limpio.get_dupes(column_names=['nombre_cliente', 'apellido_cliente'])
nulos_restantes = df_limpio.filter_on("nombre_cliente.isna() or apellido_cliente.isna()")

if duplicados_restantes.empty and nulos_restantes.empty:
    print("¡Verificación exitosa! No quedan duplicados ni nulos en las columnas de cliente.")
else:
    print("¡Error! Aún quedan datos sucios:")
    if not duplicados_restantes.empty:
        print("\nDuplicados restantes:")
        print(duplicados_restantes)
    if not nulos_restantes.empty:
        print("\nNulos restantes:")
        print(nulos_restantes)



--- 1. DataFrame Original (Sucio con Nulos y Duplicados) ---
Observa los valores 'NaN' y 'None', además de los duplicados.
    ID de Transacción nombre_cliente apellido_cliente  monto       fecha
0                 101          Maria            Gomez  100.0  2025-09-10
1                 102           Juan            Perez  150.0  2025-09-10
2                 103          Pedro            Lopez  200.0  2025-09-11
3                 104          Maria            Gomez  100.0  2025-09-10
4                 105           Juan            Perez  250.0  2025-09-12
5                 106            Ana            Silva   80.0  2025-09-12
6                 107          Maria            Gomez  100.0  2025-09-13
7                 108           Juan            Perez  150.0  2025-09-10
8                 109            NaN          Ramirez   50.0  2025-09-14
9                 110         Carlos              NaN  300.0  2025-09-15
10                111            NaN              NaN    NaN         NaN



/home/idavid/miniconda3/lib/python3.13/site-packages/pandas_flavor/register.py:161: FutureWarning: This function will be deprecated in a 1.x release. Please use `pd.DataFrame.query` instead.
  return method(self._obj, *args, **kwargs)
/home/idavid/miniconda3/lib/python3.13/site-packages/pandas_flavor/register.py:161: DeprecationWarning: This function will be deprecated in a 1.x release. Kindly use `pd.DataFrame.query` instead.
  return method(self._obj, *args, **kwargs)
